# 解答② PyTorch で 2層NN による MNIST 分類

> **このファイルは `exercises/ex_02_mnist_nn.ipynb` の解答です。**  
> 課題に取り組む前に見ないようにしてください。

---

## セットアップ

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')
print(f'PyTorch バージョン: {torch.__version__}')
torch.manual_seed(42)

---

## Step 1: MNIST データセットの読み込み

In [ ]:
data_root = '/data/shared/datasets' if os.path.exists('/data/shared') else './data'
print(f'データ保存先: {data_root}')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root=data_root, train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f'\n訓練データ: {len(train_dataset)} 枚')
print(f'テストデータ: {len(test_dataset)} 枚')

---

## Step 2: モデルの定義（解答）

```
入力: (batch, 784)
  ↓  fc1: Linear(784 → 256) + ReLU
  ↓  fc2: Linear(256 → 10)
出力: (batch, 10)
```

In [ ]:
class TwoLayerNN(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, output_dim=10):
        super().__init__()
        self.fc1  = nn.Linear(input_dim, hidden_dim)   # 解答: 第1層
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(hidden_dim, output_dim)  # 解答: 第2層

    def forward(self, x):
        x = x.view(x.size(0), -1)       # 解答: (batch, 1, 28, 28) → (batch, 784)
        x = self.relu(self.fc1(x))       # 解答: fc1 → ReLU
        x = self.fc2(x)                  # 解答: fc2
        return x

model = TwoLayerNN().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\n総パラメータ数: {total_params:,}')

---

## Step 3: 損失関数と最適化アルゴリズム

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print('損失関数: CrossEntropyLoss')
print('最適化:   Adam (lr=0.001)')

---

## Step 4: 学習ループの実装（解答）

PyTorch 学習ループの **5ステップ**:
1. 順伝播: `model(imgs)`
2. 損失計算: `criterion(outputs, labels)`
3. 勾配リセット: `optimizer.zero_grad()`
4. 逆伝播: `loss.backward()`
5. パラメータ更新: `optimizer.step()`

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """1エポック分の訓練を行い、平均 loss と accuracy を返す"""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)                    # 解答 1. 順伝播
        loss = criterion(outputs, labels)        # 解答 2. 損失計算
        optimizer.zero_grad()                    # 解答 3. 勾配リセット
        loss.backward()                          # 解答 4. 逆伝播
        optimizer.step()                         # 解答 5. パラメータ更新

        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(dim=1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """テストデータで評価し、平均 loss と accuracy を返す"""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():    # 解答: 評価時は勾配計算不要
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            correct    += (outputs.argmax(dim=1) == labels).sum().item()
            total      += imgs.size(0)

    return total_loss / total, correct / total

In [ ]:
# 実行時間: CPU で約3〜5分、GPU で約30秒

n_epochs = 5
train_losses, train_accs = [], []
test_losses,  test_accs  = [], []

for epoch in range(1, n_epochs + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)

    train_losses.append(tr_loss); train_accs.append(tr_acc)
    test_losses.append(te_loss);  test_accs.append(te_acc)

    print(f'Epoch {epoch}/{n_epochs}  '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  '
          f'test_loss={te_loss:.4f}  test_acc={te_acc:.4f}')

---

## Step 5: 学習曲線の可視化

In [ ]:
epochs = range(1, n_epochs + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(epochs, train_losses, color='#0D4A38', linewidth=2, marker='o', markersize=5, label='Train')
ax1.plot(epochs, test_losses,  color='#7C5C00', linewidth=2, marker='s', markersize=5, label='Test', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('損失曲線')
ax1.legend(); ax1.set_xticks(epochs)

ax2.plot(epochs, [a * 100 for a in train_accs], color='#0D4A38', linewidth=2, marker='o', markersize=5, label='Train')
ax2.plot(epochs, [a * 100 for a in test_accs],  color='#7C5C00', linewidth=2, marker='s', markersize=5, label='Test', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('精度曲線')
ax2.legend(); ax2.set_ylim(80, 100); ax2.set_xticks(epochs)

plt.tight_layout()
plt.show()
print(f'最終テスト精度: {test_accs[-1]*100:.2f}%')

---

## Step 6: 予測結果の確認

In [ ]:
model.eval()
imgs, labels = next(iter(test_loader))
imgs_dev = imgs.to(device)

with torch.no_grad():
    outputs = model(imgs_dev)
    preds = outputs.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].squeeze(), cmap='gray')
    color = '#0D4A38' if preds[i] == labels[i] else '#991B1B'
    ax.set_title(f'予:{preds[i].item()} 正:{labels[i].item()}', fontsize=8, color=color)
    ax.axis('off')

fig.suptitle('予測結果（緑=正解、赤=誤答）', fontsize=11)
plt.tight_layout()
plt.show()

---

## チャレンジ問題の解答: hidden_dim を変えてみよう

In [ ]:
# 実行時間: CPU で約10分（3モデル × 3エポック）、GPU で約2〜3分

hidden_dims = [64, 256, 512]
colors_exp  = ['#991B1B', '#0D4A38', '#7C5C00']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

for hd, color in zip(hidden_dims, colors_exp):
    model_exp = TwoLayerNN(hidden_dim=hd).to(device)
    opt_exp   = optim.Adam(model_exp.parameters(), lr=1e-3)
    params    = sum(p.numel() for p in model_exp.parameters())

    tr_losses_exp, te_accs_exp = [], []
    for _ in range(3):
        tr_l, _ = train_epoch(model_exp, train_loader, criterion, opt_exp, device)
        te_l, te_a = evaluate(model_exp, test_loader, criterion, device)
        tr_losses_exp.append(tr_l)
        te_accs_exp.append(te_a * 100)

    label = f'hidden={hd} ({params:,}params)'
    ax1.plot(range(1, 4), tr_losses_exp, color=color, marker='o', linewidth=2, label=label)
    ax2.plot(range(1, 4), te_accs_exp,   color=color, marker='s', linewidth=2, label=label)

for ax, title, ylabel in [(ax1, '訓練損失', 'Loss'), (ax2, 'テスト精度', 'Accuracy (%)')]:
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.set_xticks([1, 2, 3])

plt.tight_layout()
plt.show()